# Lab 02 · RAG: ground answers in retrieved knowledge

**Prepared by Mustafa Assaf** · Lecture 1: Ask, check, retrieve · ~15–20 min core work (estimate; setup and extras take longer)

## RAG: find the right microservices policy
**Start with a problem:** A team has policies for data ownership, resilience, and messaging. How can an assistant find the right policy without receiving the whole library for every question?

**In this lab:** Send the same questions to an LLM twice: first with all policies, then with only retrieved policies. Inspect the policy library, embeddings, retrieval results, prompts, and answers.

**Goal:** Learn what retrieval adds and when it can help, while reviewing microservices decisions. A retrieved policy is evidence to inspect, not proof that the answer is correct.

The library is deliberately small. Chroma and a local MiniLM embedding model run on CPU; the first run downloads the embedding model. The answers use the configured model API and its key.

**At a glance**

![Lab 02 at a glance: Policy index (embed + store) or Question (embed query), then Retrieve (top 2 policies), then Grounded answer (question + evidence), then Check citations (IDs + support).](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/02.svg)

## Think about the example

API composition asks services for data and joins their responses. CQRS maintains a separate read model for queries, which can lag behind writes. A citation identifies a source; it does not prove the answer follows from it.

**Concepts:** Database per service; API composition vs CQRS; Idempotent consumers and transactional outbox.

Use the scenario above to consider these questions before running the code:

1. Why is CQRS a better fit for searchable order history than a large in-memory API join?
2. Does citing a valid source ID prove that the source supports an answer?

## The idea in context

Sending every policy increases prompt size as a library grows. RAG retrieves selected evidence; embeddings and nearest-neighbour scores do not guarantee relevance.

Citing a valid source ID does not by itself prove factual support.

| Key term | Meaning |
| --- | --- |
| **Context window** | The maximum amount of text a model can read in one call. |
| **Embedding** | A vector of numbers that represents the meaning of a text. |
| **Vector database** | A store that finds the vectors nearest to a query vector. |
| **Grounded answer** | An answer supported by, and citing, the retrieved sources. |

## Pattern · Retrieval-Augmented Generation

*Single-agent pattern from the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide.*

| | |
| --- | --- |
| **Problem** | A model needs relevant private or changing information to answer reliably. |
| **Solution** | Retrieve useful chunks, add them to the question, and ask the model to cite them. |
| **Flow** | Question → Retrieve → Add context → Grounded answer |
| **Trade-off** | Shorter prompts may reduce input usage, but retrieval and indexing have costs and can miss relevant evidence. |
| **In this lab** | Each microservices policy is one chunk. MiniLM embeds it, Chroma retrieves candidates, and we inspect the source IDs. |

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Choose `PROVIDER` and save its key in Colab Secrets. DeepSeek uses `DEEPSEEK_API_KEY`; Gemini and FreeInference use their respective secrets. `client` stops after 12 calls; rerun setup to reset the count.

In [ ]:
from importlib.util import find_spec
from importlib.metadata import PackageNotFoundError, version
try:
    coursekit_version = version("agentic-ai-coursekit")
except PackageNotFoundError:
    coursekit_version = None
if coursekit_version != "0.5.0" or not all(find_spec(name) for name in ['coursekit', 'chromadb']):
    %pip install -q --upgrade "agentic-ai-coursekit[rag] @ https://github.com/mkassaf/agentic-ai-course/archive/fc798d4d9fda69756f7d860756fe7c0507f7f2a2.zip"
    import importlib, sys
    for name in list(sys.modules):
        if name == "coursekit" or name.startswith("coursekit."):
            del sys.modules[name]
    importlib.invalidate_caches()
from coursekit import Client, PROVIDERS, secret
PROVIDER = "deepseek"  # or "gemini" or "freeinference"
settings = PROVIDERS[PROVIDER]
client = Client(model=settings["model"], family=settings["family"],
                base_url=settings["base_url"], api_key=secret(settings["secret_name"]),
                max_calls=12, max_tokens=256, extra_body=settings.get("extra_body"))

## The two paths at a glance

```text
WITHOUT RAG, for each question:
All categories and policies ──┐
Question ─────────────────────┴──> Model ──> Answer

WITH RAG:
Policies ──> Split into chunks ──> Embed ──> Vector store
Question ──> Embed ──> Retrieve top matches ──> Model ──> Answer + policy IDs
```

Here each policy is already a small chunk. Only the RAG path retrieves: the first path sends the entire library again for each question.

### Common RAG techniques

| Technique | What it does | When it helps |
| --- | --- | --- |
| **Chunking** | Splits documents into smaller passages; overlapping chunks can retain context. | The useful rule is only part of a long document. |
| **Semantic / vector search** | Uses embeddings to find similar meanings. This lab uses it. | The question and policy use different words. |
| **Keyword search** | Finds matching words or identifiers, such as `eventId`. | Exact terms matter. |
| **Hybrid search** | Combines keyword and semantic results. | Both exact terms and meaning matter. |
| **Metadata filters** | Restrict candidates by fields such as category, version, or access permissions. | Only policies for one service or audience are applicable. |
| **Reranking** | Re-scores retrieved candidates before choosing which to send. | The first retrieval contains useful but poorly ordered results. |
| **Query rewriting** | Reformulates a question before searching. | A short or ambiguous question retrieves poor matches. |

These are options, not steps every system must use. This lab implements the vector-search path; the other techniques are here to compare and explore later. See the [original RAG paper](https://arxiv.org/abs/2005.11401) and [Google's hybrid-search explanation](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search/about-hybrid-search).

## Step 1 · Create categories and policies
Each category contains short, numbered microservices policies. Each policy will become one searchable chunk. The questions use different scenarios, including one unrelated question for inspecting retrieval limits.

In [ ]:
policies = {
    "Data ownership": [
        ("P1", "Each service owns its private database. Other services use its API or published events."),
        ("P2", "API composition joins small results by ID from services; broad searches may be expensive."),
        ("P3", "CQRS supports searchable order history with a separate read model that can lag behind writes."),
    ],
    "Resilience": [
        ("P4", "Retry a transient failure only for an idempotent operation, with a limit and backoff."),
        ("P5", "Do not blindly retry a card charge whose outcome is unknown; reconcile it first."),
        ("P6", "A bulkhead isolates resource pools so one slow dependency cannot exhaust the others."),
    ],
    "Messaging": [
        ("P7", "A consumer deduplicates an eventId with its business effect in one local transaction."),
        ("P8", "A transactional outbox stores the business change and publication intent in one local transaction."),
        ("P9", "A saga coordinates local transactions and compensates completed actions when needed."),
    ],
}
questions = ["May Reporting read the private database owned by the Order service?",
    "Support needs searchable order history by customer name across services. Which pattern fits?",
    "A payment charge timed out and may have succeeded. Should we retry it blindly?",
    "How should a consumer handle an event delivered more than once?",
    "What should we eat for lunch?"]
for category, items in policies.items():
    print(f"\n{category}:")
    print("\n".join(f"  [{pid}] {text}" for pid, text in items))

## Step 2 · Ask without RAG: send the whole library each time
Both calls send the same full policy library, even though each question needs only a small part of it. The logs show the **complete user prompt**, its word count, the model's answer, and input tokens reported by the provider (when available).

Two reasons to investigate RAG:

1. **Long context:** as a library grows, relevant details can be harder to use. The [Lost in the Middle study](https://arxiv.org/abs/2307.03172) found position-sensitive performance in some long-context tasks. This tiny example is not a test of that effect, and it does not prove your model will miss the middle.
2. **Input cost:** model services commonly charge for input tokens, so repeatedly sending more text can increase cost. Actual billing depends on the provider's tokenizer, prices, caching, and other usage. The provider-reported input tokens cover the full request, including the system message. Word counts are **not** token or dollar measurements, and token usage alone does not account for caching, output, or retrieval costs.

A model can answer well without RAG; the question is whether sending all policies remains useful as the library and request volume grow.

In [ ]:
records = [(pid, category, text) for category, items in policies.items() for pid, text in items]
all_policies = "\n".join(f"[{pid}] {category}: {text}" for pid, category, text in records)

def make_prompt(question, context):
    return (f"Question: {question}\nPolicies:\n{context}\n"
            "Use only these policies. Answer briefly and cite relevant policy IDs in square brackets.")

baseline_words, baseline_tokens = [], []
for question in questions[:2]:
    prompt = make_prompt(question, all_policies)
    baseline_words.append(len(prompt.split()))
    print("\n--- WITHOUT RAG: full prompt ---\n" + prompt)
    print("Prompt size:", len(prompt.split()), "words (not tokens)")
    answer = client.ask(prompt)
    tokens = client.trace[-1]["input_tokens"]
    baseline_tokens.append(tokens)
    print("Provider input tokens:", tokens if tokens is not None else "not reported")
    print("Model answer:", answer)

## Step 3 · Embed and store the policies
An **embedding** represents text as a vector of numbers. The same embedding model converts policies and questions into vectors so the store can compare them. The vector's individual numbers are not human-readable policy labels.

We show the vector length and first five values, then store each complete policy with its ID and category. Chroma embeds the policies during `upsert`; rerunning the cell updates the same IDs.

In [ ]:
import chromadb
from chromadb.config import Settings
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction

embed = DefaultEmbeddingFunction()
policy_vector, question_vector = embed([records[0][2], questions[0]])
print("Policy vector size:", len(policy_vector), "| first 5:", [round(float(x), 3) for x in policy_vector[:5]])
print("Question vector size:", len(question_vector), "| first 5:", [round(float(x), 3) for x in question_vector[:5]])

db = chromadb.Client(Settings(anonymized_telemetry=False))
collection = db.get_or_create_collection(
    "architecture-policies", embedding_function=embed,
    configuration={"hnsw": {"space": "cosine"}},
)
collection.upsert(ids=[pid for pid, _, _ in records],
                  documents=[text for _, _, text in records],
                  metadatas=[{"category": category} for _, category, _ in records])
print("Stored policy chunks:", collection.count())

## Step 4 · Retrieve for several questions
The retriever embeds each question and returns the **two nearest** policies. A smaller cosine distance means a closer vector match, not a guaranteed correct or applicable policy. Logs show the question, each candidate's ID, category, distance, and text.

Notice the lunch question: nearest-neighbor search still returns the closest available policies, even if none are useful. A real system needs a tested relevance or abstention policy; it should not trust a match merely because it exists.

In [ ]:
def retrieve(question):
    result = collection.query(query_texts=[question], n_results=2)
    return list(zip(result["ids"][0], result["documents"][0],
                    result["metadatas"][0], result["distances"][0]))

for question in questions:
    print("\nQUESTION:", question)
    for pid, text, metadata, distance in retrieve(question):
        print(f"  [{pid}] {metadata['category']} | distance {distance:.3f} | {text}")

## Step 5 · Ask again using only the retrieved policies
Ask the **same first two questions** with the same instructions as Step 2, but put only the two retrieved chunks in each prompt. Compare the printed prompts, word counts, and provider-reported input tokens. Some providers do not report usage; `None` means unavailable. Retrieval also consumes resources, and token counts alone cannot establish total system cost.

The code checks whether each cited ID was retrieved. A valid ID does **not** prove that a policy supports the answer; read both the answer and the policy. Results can vary with the model and embedding version.

In [ ]:
import re

for question, without_rag_words, without_rag_tokens in zip(questions[:2], baseline_words, baseline_tokens):
    hits = retrieve(question)
    context = "\n".join(f"[{pid}] {metadata['category']}: {text}"
                        for pid, text, metadata, _ in hits)
    prompt = make_prompt(question, context)
    print("\n--- WITH RAG: retrieved prompt ---\n" + prompt)
    print("Retrieved IDs:", [pid for pid, _, _, _ in hits])
    print("Prompt words: all policies =", without_rag_words, "| retrieved =", len(prompt.split()))
    answer = client.ask(prompt)
    tokens = client.trace[-1]["input_tokens"]
    print("Provider input tokens: all policies =", without_rag_tokens, "| retrieved =", tokens)
    cited = set(re.findall(r"\bP\d+\b", answer))
    retrieved = {pid for pid, _, _, _ in hits}
    print("Model answer:", answer)
    print("Citation IDs valid:", bool(cited) and cited.issubset(retrieved))

## What to look for

- Step 1 prints three categories and nine policy chunks.
- Step 2 sends all nine policies with each of two questions. Inspect the prompts and answers.
- Step 3 prints short vector samples and the number of stored chunks.
- Step 4 shows two candidates and distances for each of five questions, including the unrelated one.
- Step 5 sends only two candidates per question, prints the full prompts and compares their word counts and provider-reported input-token counts with Step 2. Check both citation IDs and whether the policy actually supports the answer.

Word counts illustrate what is sent and are not token counts. Provider-reported input tokens are usage counts, not billed amounts or evidence of a lost-in-the-middle failure. A different model may answer differently.

## Try it
1. Compare the returned policies for private databases, searchable history, uncertain card charges, and duplicate events. Is the correct policy among the two candidates?
2. Move a relevant policy to different positions in the full library and repeat the baseline. What would you need to change to test position effects fairly?
3. Change the number of retrieved chunks from 2 to 1 or 3. What changes in the prompt, evidence, and answer?
4. Search for `eventId`. Why might keyword search complement a semantic search?
5. Add a policy about mobile BFFs, run Steps 1–5 again, and inspect its category and retrieval.

## Takeaway
A full-context prompt sends every policy repeatedly. RAG first retrieves a smaller set and asks the model to use it. Both approaches need evaluation, and retrieval can return irrelevant or missing evidence.